## Goal : 


### Unify data in single datasheets (wcup_has_teams.csv and Players_Table.csv)


In [3]:
import os
import sys
import numpy as np
import pandas as pd
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.orm import declarative_base
import sys
sys.path.insert(1,'..//1 - Web Scrapping//Classes')
from classes import Worldcup,Team_Players,WorldcupPlayers,WorldcupQuarters,Team_Quarters,worldcups,name_of_all_teams,Team

In [4]:
# Change path to the root

webscrapping_path = os.path.join('..','1 - Web Scrapping')

def pjoin(name): return os.path.join(webscrapping_path, name)

with open(pjoin('name_of_all_worldcups.txt'), 'r', newline='') as f:    
    worldcups = f.read().splitlines(False)

with open(pjoin('name_of_all_teams.txt'), 'r', newline='') as f:
    name_of_all_teams = f.read().splitlines(False)

In [5]:
wcup_has_teams_df = pd.DataFrame(columns = ['YEAR', 'HOST', 'TEAM', 'TYPE', 'N_PLAYERS', 'FIRST_QUARTILE', 'SECOND_QUARTILE', 'THIRD_QUARTILE', 'FORTH_QUARTILE'])
for wcup in worldcups:
    selected_wcup = wcup
    year_wcup = pd.DataFrame([[selected_wcup.split(' ')[0],selected_wcup.split(' ')[1]],[selected_wcup.split(' ')[0],selected_wcup.split(' ')[1]]], columns=['YEAR', 'HOST'])
    wcup_df = pd.DataFrame(columns = ['YEAR', 'HOST', 'TEAM', 'TYPE', 'N_PLAYERS', 'FIRST_QUARTILE', 'SECOND_QUARTILE', 'THIRD_QUARTILE', 'FORTH_QUARTILE'])
    for team in WorldcupQuarters(selected_wcup).get_teams()[:]:
        df = WorldcupQuarters(selected_wcup).get_quarters_from_team(team).df().T
        df.reset_index(drop=False,inplace=True)
        df['TEAM'] = team
        df.columns = ['TYPE', 'N_PLAYERS', 'FIRST_QUARTILE', 'SECOND_QUARTILE', 'THIRD_QUARTILE', 'FORTH_QUARTILE','TEAM']
        df = df[['TEAM', 'TYPE', 'N_PLAYERS', 'FIRST_QUARTILE', 'SECOND_QUARTILE', 'THIRD_QUARTILE', 'FORTH_QUARTILE']]
        df_wcup_team = pd.concat([year_wcup, df], axis=1)
        wcup_df = pd.concat([wcup_df, df_wcup_team], axis=0, ignore_index=True)
    wcup_has_teams_df = pd.concat([wcup_has_teams_df, wcup_df], axis=0, ignore_index=True)

db_path = os.path.join('..','2 - Database')
def pjoin(name): return os.path.join(db_path, name)
wcup_has_teams_df.to_csv(pjoin('wcup_has_teams.csv'), index=False, encoding='utf-8-sig')

C:\Users\italo\AppData\Local\Temp\ipykernel_6804\3062795142.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wcup_df = pd.concat([wcup_df, df_wcup_team], axis=0, ignore_index=True)
C:\Users\italo\AppData\Local\Temp\ipykernel_6804\3062795142.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wcup_has_teams_df = pd.concat([wcup_has_teams_df, wcup_df], axis=0, ignore_index=True)
C:\Users\italo\AppData\Local\Temp\ipykernel_6804\3062795142.py:13: FutureWarning: The behavior of DataFrame concate

In [4]:
columns = ["Game_Year","Host","Team","Player",'Position','Day','Month','Year']
df = pd.DataFrame([],columns = columns)
for worldcup in worldcups:
        game_year = worldcup.split(" ")[0]
        host = worldcup.split(" ")[1]
        wcup = WorldcupPlayers(worldcup)
        teams = wcup.get_teams()
        for team_name in teams:
            # Takes the DataFrame with information from the players of the team
            team_df = WorldcupPlayers(worldcup).get_players_from_team(team_name).df()
            n_players = team_df.shape[0]
            team_df.index = range(1,n_players+1)

            # Insert in the database the information for every player of the team
            for player in range(1,n_players):
                players_name,position,day,month,year = team_df.loc[player].values
                if(players_name == 0 or players_name == '0'):
                    continue
                else:
                    df2 = pd.DataFrame([[game_year,host,team_name,players_name,position,day,month,year]],columns = columns)
                    df = pd.concat((df,df2),axis = 0)

In [ ]:
df.drop_duplicates(subset = ['Player','Day','Month','Year'],inplace = True)
df.index = range(1,df.shape[0]+1)
df.to_csv('Players_table.csv')